<h6><center>Big Data : Enjeux, stockage et extraction</center></h6>

<h1>
<hr style=" border:none; height:3px;">
<center>TP n°1 : Introduction à MapReduce</center>
<hr style=" border:none; height:3px;">
</h1>

**Source** : Cours CentraleSupélec de Francesca Bugiotti

Ne disposant pas d'un cluster de machines ni d'une installation locale de Hadoop MapReduce, nous nous contenterons de simuler une implémentation MapReduce en Python sur quelques exemples.

## Exercice n°1 : Compter les nombres pairs et impairs

L'objectif est de déterminer le nombre d'entiers pairs et d'entiers impairs présents dans un fichier contenant plusieurs lignes.

Implémentez une procédure MapReduce composée de :

1. Une fonction `map` qui produit une paire clé-valeur `(o,1)` pour chaque nombre impair (`o` pour "odd"), et une paire clé-valeur `(e,1)` pour chaque nombre pair (`e` pour "even").
2. Une fonction `reduce` qui additionne les valeurs associées à chaque clé.

Supposons que le fichier contienne les lignes suivantes :


```
1 2 3 3

3 4 4

5 6 7

9 8 7 4
```


### Étape 1 - Split

Chaque ligne du fichier peut être transformée en une liste d'entiers. On réalise cette étape à la main ici :

In [ ]:
fichier = [[1,2,3,3], [3,4,4], [5,6,7], [9,8,7,4]]


### Étape 2 - Map

<p align="justify">
<font size="3">   

La fonction `map` prendra en entrée cette liste d'entiers et devra générer une paire `(o,1)` pour chaque nombre impair et une paire `(e,1)` pour chaque nombre pair.

Dans notre exemple, `map()` retournera pour la première ligne :

<code>[ (o,1), (e,1), (o,1), (o,1) ]</code>

Pour la seconde ligne:

<code>[ (o,1), (e,1), (e,1) ]</code>

etc.

</font>
</p>


In [ ]:
# Map ligne

# En entrée : ligne
# En sortie : paires clé-valeur

def map_count_odd_even_line(line):
  # fonction map à appliquer à chaque ligne
  paires = [('e',1) if (n%2 == 0) else ('o',1) for n in line]
  return paires

In [ ]:
# Map fichier : version liste avec sous-listes

# En environnement distribué, les itérations de la boucle `for`
# seraient exécutées en parallèle sur plusieurs machines

def map_count_odd_even_file_no_concat(file_input):
  mapper_output = []
  for line in file_input:
    mapper_output.append(map_count_odd_even_line(line))
  return mapper_output

# Exécution

mapper_output = map_count_odd_even_file_no_concat(fichier)
print(mapper_output)

In [ ]:
# Map fichier : version liste sans sous-liste

# En environnement distribué, les itérations de la boucle `for`
# seraient exécutées en parallèle sur plusieurs machines

def map_count_odd_even_file_concat(fichier):
  mapper_output = []
  for line in fichier :
    mapper_output += map_count_odd_even_line(line)
  return mapper_output

# Exécution

mapper_output = map_count_odd_even_file_concat(fichier)
print(mapper_output)

### Étape 3 - Shuffle

Définissez une fonction qui groupera les paires obtenues à l'étape précédente par clé.

Cette fonction prendra en entrée une liste de listes de paires clé-valeur et  retournera en sortie deux listes de paires clé-valeur.  

Par exemple :    

**Entrée**  
<code>[ [(o, 1), (e, 1), (o, 1), (e, 1)], [(o, 1),(e, 1),(e, 1)], [(o, 1), (e, 1), (o,1)], [(o, 1), (e, 1), (o, 1), (e,1)] ]</code>

**Sortie**  
<code>[ [(o, 1), (o, 1), (o, 1), (o, 1), (o, 1), (o, 1), (o, 1)], [(e, 1), (e, 1), (e, 1), (e, 1), (e, 1), (e, 1), (e, 1)] ]</code>

</font>
</p>


In [ ]:
# Map : version liste avec sous-listes
# Shuffle : version `list`

# En entrée : paires clé-valeur pour chacune des lignes
# En sortie : paires clé-valeur regroupées par clé

def shuffle_count_odd_even_list(mapper_output):
    odd = []
    even = []
    for line in mapper_output:
        odd += [pair for pair in line if pair[0] == 'o']
        even += [pair for pair in line if pair[0] == 'e']
    shuffler_output = [odd,even]
    return shuffler_output

# Exécution

shuffler_output = shuffle_count_odd_even_list(map_count_odd_even_file_no_concat(fichier))
print(shuffler_output)

In [ ]:
# Map : version liste sans sous-liste
# Shuffle : version `dict`

# En entrée : paires clé-valeur
# En sortie : paires clé-valeur regroupées par clé

def shuffle_count_odd_even_dict(mapper_output):
    # création d'un dictionnaire vide
    shuffler_output = {}
    for paire in mapper_output :
        if paire[0] in shuffler_output.keys():
            # la clé a déjà été ajoutée au dictionnaire
            shuffler_output[paire[0]].append(paire[1])
        else:
            # la clé n'a pas encore été ajoutée au dictionnaire
            shuffler_output[paire[0]] = [paire[1]]
    return shuffler_output

# Exécution

shuffler_output = shuffle_count_odd_even_dict(map_count_odd_even_file_concat(fichier))
print(shuffler_output)

### Étape 4 - Reduce

Définissez une fonction qui produira le résultat final à partir des deux listes obtenues à l'étape précédente. La fonction doit être appliquée à chaque clé l'une après l'autre (donc ici : deux fois).

In [ ]:
# Map : version liste sans sous-liste
# Shuffle : version `dict`
# Reduce : une clé à la fois

# En entrée : paires clé-valeur regroupées dans un dictionnaire
# En sortie : paires clé-valeur traitées

def reduce_count_odd_even(shuffler_output):
    reducer_output = (shuffler_output[0],sum(shuffler_output[1]))
    return reducer_output

# Exécution

shuffler_output = shuffle_count_odd_even_dict(map_count_odd_even_file_concat(fichier))

# En environnement distribué, les itérations de la boucle `for`
# seraient exécutées en parallèle sur plusieurs machines

map_reduce_output = []
for item in shuffler_output.items():
    map_reduce_output.append(reduce_count_odd_even(item))

print('MapReduce :', map_reduce_output)

In [ ]:
# Map : version liste avec sous-listes
# Shuffle : version `list`
# Reduce : une clé à la fois

# En entrée : paires clé-valeur regroupées dans une liste
# En sortie : paires clé-valeur traitées

def reduce_count_odd_even(shuffle_output):
    key = shuffle_output[0][0]
    value = sum([p[1] for p in shuffle_output])
    return (key, value)

# Exécution

shuffler_output = shuffle_count_odd_even_list(map_count_odd_even_file_no_concat(fichier))

# En environnement distribué, les itérations de la boucle `for`
# seraient exécutées en parallèle sur plusieurs machines

map_reduce_output = []
for shuffler_group in shuffler_output:
  map_reduce_output.append(reduce_count_odd_even(shuffler_group))

print('MapReduce :', map_reduce_output)

In [ ]:
# Map : version liste avec sous-listes
# Shuffle : version `list`
# Reduce : toutes les clés à la fois, version plus éloignée de la logique MapReduce

# En entrée : paires clé-valeur regroupées dans des listes
# En sortie : paires clé-valeur traitées

def reduce_count_odd_even(shuffle_output):
    for pair in shuffle_output:
        if pair[0][0] == 'o':
            odd = ('o', sum([p[1] for p in pair]))
        else:
            even = ('e', sum([p[1] for p in pair]))
    return odd, even

# Exécution

shuffler_output = shuffle_count_odd_even_list(map_count_odd_even_file_no_concat(fichier))

map_reduce_output = reduce_count_odd_even(shuffler_output)

print('MapReduce :', map_reduce_output)

## Exercice n°2 : Réaliser un index inversé

Dans cet exercice, l'objectif est de créer un index inversé. Un index inversé est un index qui associe chaque mot à la liste des noms de fichiers dans lesquels le mot apparaît.

Vous utiliserez `books.json`, qui contient plusieurs lignes. Chaque ligne représente un fichier différent contenant un livre. La fonction pour l'étape Split est la suivante :

In [ ]:
import json

In [ ]:
def read_data(filename):
    with open(filename, mode='r', encoding='utf-8') as file:
        for line in file:
            record = json.loads(line)
            yield(record)
print(list(read_data('books.json')))

Le résultat est une liste de listes. Chaque sous-liste contient deux éléments : le nom du fichier et une chaîne de caractères composée du titre et du contenu du livre.  

Le résultat attendu de la procédure MapReduce est une liste contenant un ensemble d'éléments (w, L), où w est un mot apparaissant dans au moins l'un des textes et L est la liste des noms des fichiers contenant w (sans répétition).

Décrivez une implémentation possible de la fonction :
1. `map()` qui, à partir du texte, produit un ensemble de paires clé-valeur
2. `shuffle()` qui regroupe les paires clé-valeur selon un critère
3. `reduce()` qui produit le résultat final  

en détaillant toutes les actions que les fonctions `map()` et `reduce()` doivent effectuer (commentez le code).

In [ ]:
import re

In [ ]:
# 1. Map

# 1.1 Map ligne

# En entrée : ligne
# En sortie : paires clé-valeur

def mapper_inv_index_line(line):
    filename = line[0]
    # Transformer la ligne en une liste de mots
    words = line[1].split()
    # Créer une paire par mot : (mot, nom_du_fichier)
    paires = [(word,filename) for word in words]
    return paires

# 1.2 Map fichier

# En environnement distribué, les itérations de la boucle `for`
# seraient exécutées en parallèle sur plusieurs machines

def mapper_inv_index_file(file_input):
    mapper_output = []
    for line in file_input:
        mapper_output += mapper_inv_index_line(line)
    return mapper_output

# 2. Shuffle

def shuffler_inv_index(mapper_output):
    # Création d'un dictionnaire vide
    shuffler_output = {}
    for paire in mapper_output:
        if paire[0] in shuffler_output.keys():
            # Le mot a déjà été ajouté au dictionnaire
            shuffler_output[paire[0]].append(paire[1])
        else:
            # Le mot n'a encore jamais été ajouté au dictionnaire
            shuffler_output[paire[0]] = [paire[1]]
    return shuffler_output

# 3. Reduce

# En entrée : paires clé-valeur regroupées dans un dictionnaire
# En sortie : paires clé-valeur traitées

# La sortie du shuffle a déjà la forme attendue (w, L)
# Il nous faut seulement supprimer les doublons dans L (avec `set`)

def reducer_inv_index(shuffler_output):
    # Ne conserver que les valeurs uniques pour les noms de fichier
    key = shuffler_output[0]
    value = list(set(shuffler_output[1]))
    return (key, value)



In [ ]:
file_input = list(read_data('books.json'))

shuffler_output = shuffler_inv_index(mapper_inv_index_file(file_input))

# En environnement distribué, les itérations de la boucle `for`
# seraient exécutées en parallèle sur plusieurs machines

map_reduce_output = []
for item in shuffler_output.items():
    map_reduce_output.append(reducer_inv_index(item))

print('MapReduce :')
for i in range(10):
    print(map_reduce_output[i])

Quelles améliorations pourriez-vous faire ? Modifiez votre code en fonction.

Le code actuel compte comme des mots les signes de ponctuation (crochets, parenthèses ...). Les lettres isolées sont aussi incluses. Plusieurs mots apparaissent plusieurs fois parce que la casse (majuscule / minuscule) est différente (ex : "Of", "of"). On pourrait réaliser un nettoyage à l'étape Map pour exclure la ponctuation, les mots de moins de deux lettres et mettre tous les mots en minuscule.

In [ ]:
import re

def mapper_inv_index_line_re(line):
    filename = line[0]
    # Supprimer tous les caractères qui ne sont pas des lettres
    words = re.sub(r'[^a-zA-Z]',' ',line[1].strip()).strip()
    # Transformer les espaces multiples en espace simple
    words = re.split(r'\s+',words)
    # Mettre les mots en minuscule
    # Ne garder que les mots de plus de trois lettres
    paires = [(word.lower(),filename) for word in words if len(word) >= 3]
    return paires

def mapper_inv_index_file_re(file_input):
    mapper_output = []
    for line in file_input:
        mapper_output += mapper_inv_index_line(line)
    return mapper_output

In [ ]:
file_input = list(read_data('books.json'))

mapper_output = mapper_inv_index_file_re(file_input)

print('Map :')
for i in range(10):
    print(mapper_output[i])
print()

shuffler_output = shuffler_inv_index(mapper_output)

map_reduce_output = []
for item in shuffler_output.items():
    map_reduce_output.append(reducer_inv_index(item))

print('MapReduce :')
for i in range(10):
    print(map_reduce_output[i])

## Exercice n°3 : Trouver le message le plus long

Pensant y trouver d'éventuelles axes d'amélioration pour votre entreprise, vous souhaiteriez consulter l'un des mails les plus longs envoyé au service client.

Concevez et implémentez la procédure MapReduce adéquate. Utilisez le fichiez `mails.txt`.

Pour l'étape Split, utilisez `open()` et identifiez les caractères permettant de séparer les mails les uns des autres.  

* **Split** : mail par mail (et non pas ligne par ligne cette fois)
* **Map** : paires `("MAX", (LONGUEUR(mail),mail))`
* **Combiner** (optionnel) : maximum local, une seule paire `("MAX",(MAX LONGUEUR(mail),mail))`
* **Shuffle** : `("MAX", [(LONGUEUR(mail1),mail1)),...])`  
* **Reduce** : maximum global, une seule paire `("MAX",(MAX(longueur),mail))`

In [ ]:
# 1. Split

# Décomposition du traitement en plusieurs tâches
# qui, en environnement distribué, seraient exécutées
# en parallèle sur plusieurs machines

with open('mails.txt','r') as f:
   fichier = f.read()

fichier = fichier.split('"\n')

# 2. Map

# En sortie : (MAX,(longueur, mail))

def mapper_max_mail(mail):
    return ("MAX",(len(mail),mail))

# 3. Shuffle

def shuffle_max_mail(mapper_output):
    shuffler_output = {}
    for paire in mapper_output:
        if paire[0] in shuffler_output.keys():
            shuffler_output[paire[0]].append(paire[1])
        else:
            shuffler_output[paire[0]] = [paire[1]]
    return shuffler_output

# 4. Reduce

def reducer_inv_index(shuffler_output):
    # Trier par ordre décroissant en fonction de la longueur du mail
    # Récupérer le mail associé à la première paire, correspondant au mail le plus long
    reduce_output = (shuffler_output[0], sorted(shuffler_output[1], key=lambda paire: paire[0],reverse=True)[0][1])
    return reduce_output

In [ ]:
# Exécution

# 1. Map

# En environnement distribué, les itérations de la boucle `for`
# seraient exécutées en parallèle sur plusieurs machines

mapper_output = []
for mail in fichier :
    mapper_output.append(mapper_max_mail(mail))

# 2. Shuffle

shuffler_output = shuffle_max_mail(mapper_output)

# 3. Reduce

# En environnement distribué, les itérations de la boucle `for`
# seraient exécutées en parallèle sur plusieurs machines

map_reduce_output = []
for item in shuffler_output.items():
    map_reduce_output.append(reducer_inv_index(item))

print('MapReduce :')
print(map_reduce_output)

## Exercice n°4 : Trouver les amis en commun

**L'énoncé était incorrect ! Modifications en gras. Le fichier `amis.txt` également.**

Vous administrez un réseau social comportant des millions d'utilisateurs. Pour chaque utilisateur, vous pouvez obtenir via une requête SQL la liste des utilisateurs avec qui il est ami sur le réseau.

Lorsqu'un utilisateur va sur la page d'un **ami**, vous souhaiteriez afficher le message « Vous avez N amis en commun ».
Vous ne pouvez pas effectuer une série de requêtes SQL à chaque fois que la page est accédée (trop lourd en traitement). Développez un programme MapReduce pour réaliser cette opération, que vous pourrez exécuter toutes les nuits sur votre base de données.

1) Commencez par implémenter une procédure MapReduce qui renvoie la liste des amis en communs pour chaque paire d'**amis**

2) Proposez ensuite une nouvelle fonction Reduce pour ne renvoyer que le nombre d'amis en commun

Utilisez le fichier `amis.txt` et détaillez la procédure MapReduce avant de l'implémenter.

**Question n°1**

* **Split** : ligne par ligne  
```
User => Friend_1 Friend_2 ...
```
* **Map** : paires  
```
((User,Friend_i),[Friend_1,Friend_2,...])  
```
en triant `(User,Friend_i)` dans l'ordre alphabétique, de façon à ce que les valeurs associées à `(User,Friend_i)` et `(Friend_i,User)` soient regroupées ensemble à l'étape suivante.
* **Shuffle** :   
```
((User,Friend_i),[Group_Friends_User,Group_Friends_Friend_i])
```
* **Reduce** :   
```
((User,Friend_i), INTERSECTION(Group_Friends_User,Group_Friends_Friend_i))
```

In [ ]:
# 1. Split

with open('amis.txt','r') as f:
   fichier = f.read()

fichier = fichier.split('\n')

# 2. Map

def mapper_mutual_friends(line):
    line = line.split(' => ')
    user = line[0]
    friends = line[1].split()
    # Trier dans l'ordre alphabétique le nom de l'utilisateur et de l'ami
    # Transformer la `list` d'amis en `set`
    mapper_output = [(sorted((user,f)),set(friends)) for f in friends]
    return mapper_output

# 3. Shuffle

def shuffle_mutual_friends(mapper_output):
    shuffler_output = {}
    for paire in mapper_output:
        key = tuple(paire[0])
        if key in shuffler_output.keys():
            shuffler_output[key].append(paire[1])
        else:
            shuffler_output[key] = [paire[1]]
    return shuffler_output

# 4. Reduce

def reducer_mutual_friends(shuffler_output):
    key = shuffler_output[0]
    # Calculer l'intersection des `set` d'amis des deux amis
    value = shuffler_output[1][0].intersection(shuffler_output[1][1])
    return (key, value)

In [ ]:
# Execution

# 1. Map

mapper_output = []
for line in fichier :
    mapper_output += mapper_mutual_friends(line)

# 2. Shuffle

shuffler_output = shuffle_mutual_friends(mapper_output)

# 3. Reduce

map_reduce_output = []
for item in shuffler_output.items():
    map_reduce_output.append(reducer_mutual_friends(item))

print('MapReduce :')
for friends in map_reduce_output[:5]:
  print(friends)

**Question n°2**

* **Split** : ligne par ligne  
```
User => Friend_1 Friend_2 ...
```
* **Map** : paires  
```
((User,Friend_i),[Friend_1,Friend_2,...])
```
avec `(User,Friend_i)` ou `(Friend_i,User)` en fonction de l'ordre alphabétique
* **Shuffle** :   
```
((User,Friend_i),[Group_Friends_User,Group_Friends_Friend_i])
```
* **Reduce** :   
```
((User,Friend_i), LEN(INTERSECTION(Group_Friends_User,Group_Friends_Friend_i)))
```

In [ ]:
# 1. Split

with open('amis.txt','r') as f:
   fichier = f.read()

fichier = fichier.split('\n')

# 2. Map

def mapper_mutual_friends(line):
    line = line.split(' => ')
    user = line[0]
    friends = line[1].split()
    # Trier dans l'ordre alphabétique le nom de l'utilisateur et de l'ami
    # Transformer la liste d'amis en `set`
    mapper_output = [(sorted((user,f)),set(friends)) for f in friends]
    return mapper_output

# 3. Shuffle

def shuffle_mutual_friends(mapper_output):
    shuffler_output = {}
    for paire in mapper_output:
        key = tuple(paire[0])
        if key in shuffler_output.keys():
            shuffler_output[key].append(paire[1])
        else:
            shuffler_output[key] = [paire[1]]
    return shuffler_output

# 4. Reduce

def reducer_mutual_friends(shuffler_output):
    key = shuffler_output[0]
    # Calculer l'intersection des `set` d'amis des deux amis
    # Calculer la longueur du `set` obtenu
    value = shuffler_output[1][0].intersection(shuffler_output[1][1])
    return (key, len(value))

In [ ]:
# Exécution

# 1. Map

mapper_output = []
for line in fichier :
    mapper_output += mapper_mutual_friends(line)

# 2. Shuffle

shuffler_output = shuffle_mutual_friends(mapper_output)

# 3. Reduce

map_reduce_output = []
for item in shuffler_output.items():
    map_reduce_output.append(reducer_mutual_friends(item))

print('MapReduce :')
for friends in map_reduce_output[:5]:
  print(friends)

## Bonus pour s'entraîner : Compter le nombre d'entiers

Comptez le nombre total d'entier dans le fichier suivant à l'aide d'un algorithme MapReduce. Commentez le code.

```
1  2  4.3  3

4  5  1.2  6.2  7

8  9  5.2  1  2  3

1  2

1.4  8  9.76

```

In [ ]:
fichier = [[1, 2, 4.3, 3], [4, 5, 1.2, 6.2, 7], [8, 9, 5.2, 1, 2, 3], [1, 2], [1.4, 8, 9.76]]

In [ ]:
# 1. Split

# Déjà fait : découpage par ligne

# 2. Map

def mapper_count_integer(line):
    mapper_output = []
    for number in line:
       if int(number) == number:
          mapper_output.append(('integer',1))
    return mapper_output

# 3. Shuffle

def shuffle_count_integer(mapper_output):
    shuffler_output = {}
    for paire in mapper_output:
        if paire[0] in shuffler_output.keys():
            shuffler_output[paire[0]].append(paire[1])
        else:
            shuffler_output[paire[0]] = [paire[1]]
    return shuffler_output

# 4. Reduce

def reducer_count_integer(shuffler_output):
    reducer_output = (shuffler_output[0],sum(shuffler_output[1]))
    return reducer_output

In [ ]:
# Exécution

mapper_output = []
for line in fichier :
    mapper_output += mapper_count_integer(line)

shuffler_dict_output = shuffle_count_integer(mapper_output)

map_reduce_output = []
for item in shuffler_dict_output.items():
    map_reduce_output.append(reducer_count_integer(item))

print('MapReduce :', map_reduce_output)